In [ ]:
import pandas as pd
import kagglehub
import os
import re

# Download latest version
path = kagglehub.dataset_download("ulrikthygepedersen/online-retail-dataset")
print("Path to dataset files:", path)

In [ ]:
files = os.listdir(path)
print(files)

In [ ]:
file_path = os.path.join(path, files[0])
df = pd.read_csv(file_path)

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df.describe()

In [ ]:
df.head()
df.sample(5)

In [ ]:
def clean_columns(df):
    def convert(name):
        # Add underscore between lower-to-upper case transitions
        name = re.sub(r'(?<=[a-z])(?=[A-Z])', '_', name)
        # Handle acronym endings like ID
        name = re.sub(r'(?<=[A-Z])(?=[A-Z][a-z])', '_', name)
        return name.lower().strip()

    df.columns = [convert(col) for col in df.columns]
    return df

df = clean_columns(df)
print(df.columns)

In [ ]:
df['description'] = (
    df['description']
    .str.replace('\xa0', ' ', regex=False)   # remove non-breaking space
    .str.replace(r'\s+', ' ', regex=True)    # remove multiple spaces
    .str.strip()
    .str.title() # standardise
)

In [ ]:
df.isnull().sum()

In [ ]:
df.memory_usage(deep=True)

In [ ]:
df['stock_code'] = df['stock_code'].astype('category')
df['country'] = df['country'].astype('category')

In [ ]:
df.memory_usage(deep=True)

In [ ]:
df['customer_id'] = df['customer_id'].astype('Int64')
df['invoice_date'] = pd.to_datetime(df['invoice_date'])
df['year'] = df['invoice_date'].dt.year # date labels for pivoting
df['month'] = df['invoice_date'].dt.month
df['year_month'] = df['invoice_date'].dt.to_period('M').astype(str)
df['weekday'] = df['invoice_date'].dt.day_name()

df['is_guest'] = df['customer_id'].isnull()

print(df['is_guest'].value_counts())
print(df['is_guest'].sum())

In [ ]:
df['country'].unique()

In [ ]:
df['customer_id'].nunique()

In [ ]:
df[df.duplicated(keep=False)].sort_values('invoice_no').head(20)

In [ ]:
df = df.drop_duplicates()
df.duplicated().sum()

In [ ]:
df[df['quantity'] < 0]

In [ ]:
df['revenue'] = df['quantity'] * df['unit_price']

In [ ]:
df['is_cancelled'] = df['invoice_no'].astype(str).str.startswith('C') # flag cancelled orders

df['is_financial_adjustment'] = df['description'].str.contains(
    'adjust|debt|discount|bank|charge|manual|credit|sample',
    case=False,
    na=False
) # Identify non products / financial entries

df['is_alpha_stock'] = df['stock_code'].astype(str).str.isalpha() # identify alpha stock code

df['is_non_product'] = df['is_financial_adjustment'] | df['is_alpha_stock'] # flag all non products

In [ ]:
df['is_return'] = (
    (df['quantity'] < 0) &
    (~df['is_non_product'])
) # flag for negative product movement but not negative accountancy entry

In [ ]:
df['is_sale'] = (
    (df['quantity'] > 0) &
    (~df['is_non_product']) &
    (~df['is_cancelled'])
)

In [ ]:
df['return_value'] = df['revenue'].where(df['revenue'] < 0, 0).abs()

df['gross_sale_value'] = df['revenue'].where(df['revenue'] > 0, 0)


In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
# Check for negative unit prices
df[df['unit_price'] < 0]

In [ ]:
# Check for zero quantity sales
df[df['quantity'] == 0]

In [ ]:
df['stock_code'].value_counts()

In [ ]:
df[['is_sale', 'is_return', 'is_cancelled', 'is_non_product']].sum()

In [ ]:
print(df.groupby('is_non_product')['revenue'].sum())
print(df['revenue'].sum())

In [ ]:
# Ensure sales and returns do not overlap
print((df['is_sale'] & df['is_return']).sum())
# Ensure non-products are not classified as sale/return
print((df['is_non_product'] & df['is_sale']).sum())
print((df['is_non_product'] & df['is_return']).sum())

In [ ]:
df.to_csv("cleaned_online_retail.csv", index=False)